In [27]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, RobustScaler,LabelEncoder
import joblib

In [28]:
train_df = pd.read_csv("../../data/split/train.csv")
val_df = pd.read_csv("../../data/split/validation.csv")
test_df = pd.read_csv("../../data/split/test.csv")
x_train = train_df.drop("diabetes", axis=1)
y_train = train_df["diabetes"]

x_val = val_df.drop("diabetes", axis=1)
y_val = val_df["diabetes"]

x_test = test_df.drop("diabetes", axis=1)
y_test = test_df["diabetes"]





### convert gender to 0 and 1 using binary encoding
#### Female = 0
#### Male = 1

In [29]:
x_train["gender"] = x_train["gender"].replace({"Female": 0, "Male": 1})
x_val["gender"] = x_val["gender"].replace({"Female": 0, "Male": 1})
x_test["gender"] = x_test["gender"].replace({"Female": 0, "Male": 1})



### Engineered Features Description

- **glucose_hba1c_interaction**: Captures the combined effect of current blood glucose level and long-term glucose control (HbA1c), which may better reflect diabetes risk than either feature alone.

- **age_hba1c_interaction**: Represents how the effect of HbA1c may vary with age, helping the model capture age-related diabetes risk patterns.

- **age_bmi_interaction**: Combines age and BMI to reflect how body weight impact on diabetes risk can change across different ages.

- **bmi_hba1c_interaction**: Combines BMI and HbA1c to capture the joint effect of body weight and long-term blood sugar control.

- **age_glucose_interaction**: Combines age and blood glucose level to reflect how glucose-related risk may become more significant across older age groups.

- **high_hba1c_flag**: Binary indicator showing whether HbA1c is in the diabetic range (>= 6.5).

- **senior_flag**: Binary indicator showing whether the person is 60 years or older, since diabetes risk generally increases with age.

- **cardio_risk_flag**: Binary indicator showing whether the person has hypertension or heart disease, capturing cardiovascular-related health risk.

In [ ]:
def add_engineered_features(df):    
    df["glucose_hba1c_interaction"] = df["blood_glucose_level"] * df["HbA1c_level"]
    df["age_hba1c_interaction"] = df["age"] * df["HbA1c_level"]
    df["age_bmi_interaction"] = df["age"] * df["bmi"]
    df["bmi_hba1c_interaction"] = df["bmi"] * df["HbA1c_level"]
    df["age_glucose_interaction"] = df["age"] * df["blood_glucose_level"]
    df["high_hba1c_flag"] = (df["HbA1c_level"] >= 6.5).astype(int)
    df["senior_flag"] = (df["age"] >= 60).astype(int)

    df["cardio_risk_flag"] = ((df["hypertension"] == 1) | (df["heart_disease"] == 1)).astype(int)

    return df

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("smoking_ohe", OneHotEncoder(handle_unknown="ignore"), ["smoking_history"]),
        ("age_minmax", MinMaxScaler(), ["age"]),
        ("robust_features", RobustScaler(), ["bmi",
                                              "HbA1c_level",
                                              "blood_glucose_level",
                                              "glucose_hba1c_interaction",
                                              "age_hba1c_interaction",
                                              "age_bmi_interaction",
                                              "bmi_hba1c_interaction",
                                              "age_glucose_interaction"])
    ],
    remainder="passthrough", # to keep the rest of the columns unchanged
    verbose_feature_names_out=False  # to prevent adding the transformer name as a prefix to the feature names
    # smoking_history_No Info insted of smoking_ohe__smoking_history_No Info
)
joblib.dump(preprocessor, "preprocessor.pkl")

x_train_processed = preprocessor.fit_transform(x_train) # returns numpy array not Pandas DataFrame so we need to convert it back to DataFrame
joblib.dump(preprocessor, "preprocessor.pkl")
feature_names = preprocessor.get_feature_names_out()
x_train_processed = pd.DataFrame(
    x_train_processed,
    columns=feature_names,
    index=x_train.index
)
x_train_processed.head()


,smoking_history_No Info,smoking_history_current,smoking_history_ever,smoking_history_former,smoking_history_never,smoking_history_not current,age,bmi,HbA1c_level,blood_glucose_level,...,age_hba1c_interaction,age_bmi_interaction,bmi_hba1c_interaction,age_glucose_interaction,gender,hypertension,heart_disease,high_hba1c_flag,senior_flag,cardio_risk_flag
0,0.0,0.0,1.0,0.0,0.0,0.0,0.361862,-0.795349,0.142857,-0.847458,...,-0.239693,-0.500569,-0.251066,-0.512287,0,0,0,0,0,0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.261762,0.0,0.0,-0.847458,...,-0.489933,-0.563572,0.157075,-0.648393,0,0,0,0,0,0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.724725,-0.682171,0.142857,-0.932203,...,0.594439,0.118586,-0.180452,-0.073724,0,0,0,0,0,0
3,0.0,0.0,0.0,0.0,1.0,0.0,0.712212,0.0,0.857143,0.0,...,0.838926,0.324306,0.685615,0.502836,1,1,0,1,0,1
4,1.0,0.0,0.0,0.0,0.0,0.0,0.436937,0.0,-1.642857,-0.237288,...,-0.486577,-0.218286,-0.855959,-0.172023,0,0,0,0,0,0


## Transform Validation data 

In [36]:
x_val = add_engineered_features(x_val)
preprocessor = joblib.load("preprocessor.pkl")
x_val_processed = preprocessor.transform(x_val)
x_val_processed = pd.DataFrame(
    x_val_processed,
    columns=feature_names,   # same feature names from train
    index=x_val.index
)

x_val_processed.head()

,smoking_history_No Info,smoking_history_current,smoking_history_ever,smoking_history_former,smoking_history_never,smoking_history_not current,age,bmi,HbA1c_level,blood_glucose_level,...,age_hba1c_interaction,age_bmi_interaction,bmi_hba1c_interaction,age_glucose_interaction,gender,hypertension,heart_disease,high_hba1c_flag,senior_flag,cardio_risk_flag
0,0.0,0.0,0.0,0.0,1.0,0.0,0.612112,0.051163,0.285714,0.0,...,0.38255,0.141597,0.366241,0.291115,1,0,0,0,0,0
1,0.0,0.0,0.0,0.0,1.0,0.0,0.774775,0.787597,0.0,-0.677966,...,0.650048,0.731954,0.632091,0.166352,0,0,0,0,1,0
2,0.0,0.0,0.0,0.0,1.0,0.0,0.336837,-0.186047,0.571429,0.322034,...,-0.219559,-0.444842,0.38175,-0.19414,0,0,0,1,0,0
3,0.0,1.0,0.0,0.0,0.0,0.0,0.762262,1.989147,2.142857,0.254237,...,1.499521,1.129482,3.29865,0.781664,1,0,0,1,1,0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.787287,-0.544186,1.714286,2.033898,...,1.402685,0.272659,0.750135,2.090737,1,0,1,1,1,1


In [38]:
x_train_processed.to_csv("../../data/preprocessed/train_preprocessed.csv", index=False)
x_val_processed.to_csv("../../data/preprocessed/validation_preprocessed.csv", index=False)
